In [1]:
from runtime_support import (
    setup_client_from_env,
    build_fleet_object,
    )

import asyncio

from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

with setup_client_from_env() as client:
        fleet_api = FleetApi(client)
        agents_api = AgentsApi(client)
        systems_api = SystemsApi(client)

from core_helpers import (load_initial_fleet_state,
                          init_world_state,
                          load_initial_waypoint_state,
                          get_wps_by_trait,
                          all_wp_visitor,
                          api_navigate_ship,
                          api_get_ship_nav,
                          plot_route_svg,
                          patrol_markets,
                          )


#-------- write fleet activity and fleet specifications to db -------------
state = await load_initial_fleet_state(fleet_api)

#-------- write waypoint reference data and traits specifications to db and local objects -------------
wp_refs, wp_traits = await load_initial_waypoint_state(agents_api, systems_api)
print(f"Persisted {len(wp_refs)} waypoint refs and {len(wp_traits)} traits.")

# --------- create locally persisting objects for fleet and world states -------------------
ships_activity_obj = await build_fleet_object(fleet_api)
world_state = await init_world_state(fleet_api, agents_api, systems_api)

# --------- define ship roles -----------

# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

# manually assign ship names by role
command_ship = get_symbol_by_role(state.specs, "COMMAND")
satellite = get_symbol_by_role(state.specs, "SATELLITE")


# ---------- get market waypoints from world state ----------
# all traits of every waypoint
traits_list = world_state.traits.by_wp
# selecting MARKETPLACE trait-bearing waypoints only
markets_df = await get_wps_by_trait(traits_list, "MARKETPLACE")

shipyards_df = await get_wps_by_trait(traits_list, "SHIPYARD")
print(shipyards_df)


Persisted 89 waypoint refs and 239 traits.
[BOOT] Adapted 2 ships into fleet_object
     waypoint    x   y    distance
0   X1-XG6-A2  -23   6   23.769729
1  X1-XG6-H57   17 -42   45.310043
2  X1-XG6-C46  140  64  153.935051


In [2]:
nav_resp = await api_get_ship_nav(fleet_api, satellite)
nav_resp

ShipNav(system_symbol='X1-XG6', waypoint_symbol='X1-XG6-A2', route=ShipNavRoute(destination=ShipNavRouteWaypoint(symbol='X1-XG6-A2', type=<WaypointType.MOON: 'MOON'>, system_symbol='X1-XG6', x=-23, y=6), origin=ShipNavRouteWaypoint(symbol='X1-XG6-H57', type=<WaypointType.MOON: 'MOON'>, system_symbol='X1-XG6', x=17, y=-42), departure_time=datetime.datetime(2025, 9, 19, 14, 7, 59, 918000, tzinfo=TzInfo(UTC)), arrival=datetime.datetime(2025, 9, 19, 14, 11, 6, 917000, tzinfo=TzInfo(UTC))), status=<ShipNavStatus.IN_ORBIT: 'IN_ORBIT'>, flight_mode=<ShipNavFlightMode.CRUISE: 'CRUISE'>)

In [3]:
from market_runtime import capture_shipyard_for_waypoint, shipyard_to_db

waypoint_symbol = 'X1-XG6-A2'

await shipyard_to_db(waypoint_symbol)


ERROR! Session/line number was not unique in database. History logging moved to new session 156


In [13]:

from openapi_client.models.get_shipyard200_response import GetShipyard200Response

json.dumps(dto.to_dict())

#pprint.pprint(dto.model_dump(by_alias=True))
#await shipyard_to_db('X1-XG6-A2')

'{"symbol": "X1-XG6-A2", "shipTypes": [{"type": "SHIP_PROBE"}, {"type": "SHIP_LIGHT_SHUTTLE"}, {"type": "SHIP_LIGHT_HAULER"}], "transactions": [], "ships": [{"type": "SHIP_PROBE", "name": "Probe Satellite", "description": "A small, unmanned spacecraft that can be launched into orbit to gather data and perform basic tasks.", "activity": "GROWING", "supply": "HIGH", "purchasePrice": 22288, "frame": {"symbol": "FRAME_PROBE", "name": "Probe", "condition": 1, "integrity": 1, "description": "A small, unmanned spacecraft used for exploration, reconnaissance, and scientific research.", "moduleSlots": 0, "mountingPoints": 0, "fuelCapacity": 0, "requirements": {"power": 1, "crew": 0}, "quality": 1}, "reactor": {"symbol": "REACTOR_SOLAR_I", "name": "Solar Reactor I", "condition": 1, "integrity": 1, "description": "A basic solar power reactor, used to generate electricity from solar energy.", "powerOutput": 3, "requirements": {"crew": 0}, "quality": 1}, "engine": {"symbol": "ENGINE_IMPULSE_DRIVE_I

In [ ]:

# from selected ships' starting waypoint, draw a route that will visit all market waypoints

nav_resp = await api_get_ship_nav(fleet_api, satellite)

print(nav_resp.waypoint_symbol)
if __name__ == "__main__":
    
    start = shipyards_df.iloc[0,0]

    route = all_wp_visitor(
        shipyards_df,
        start_waypoint=start,
        return_to_start=True,   # set True to draw a loop back to start
        improve_2opt=True,
        fix_start_anchor=True,   # keep chosen start as first
    )

    print(route)
    print("Total travel distance:", route["cumulative_distance"].iloc[-1])

    plot_route_svg(
        markets_df=markets_df,
        route_df=route,
        start_waypoint=start,
        svg_path="markets_route.svg",
    )
# ---------- run ----------
async def main():
    patroller = asyncio.create_task(patrol_markets(satellite, route))
    #patroller2 = asyncio.create_task(patrol_markets(command_ship, route))
    await asyncio.gather(patroller)

# await api_navigate_ship(fleet_api, ship_symbol, wp)
await main()

X1-XG6-A2
   visit_idx    waypoint    x   y  leg_distance  cumulative_distance
0          0   X1-XG6-A2  -23   6         0.000                0.000
1          1  X1-XG6-H57   17 -42        62.482               62.482
2          2  X1-XG6-C46  140  64       162.373              224.855
3          3   X1-XG6-A2  -23   6       173.012              397.867
Total travel distance: 397.867
Saved SVG -> markets_route.svg
['X1-XG6-A2', 'X1-XG6-H57', 'X1-XG6-C46']
[PATROL] DDDD-2 looping through 3 markets.
[PATROL] -> Navigating to X1-XG6-A2 (#1/3)
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Ship is already at the destination
Prep complete
[SKIP] Navigation aborted, already at destination
[MARKET] Capturing X1-XG6-A2 …
[PATROL] -> Navigating to X1-XG6-H57 (#2/3)
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Prep complete
DDDD-2  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 187.

In [2]:
import asyncio
from core_helpers import build_nodes_from_traits_dict
from refuel_routing import plan_route_and_refuel
from typing import Tuple

async def long_dist_travel(cur_wp, dest_wp, trav_dist, max_trav_dist):

    with setup_client_from_env() as client:
        fleet_api = FleetApi(client)
        agents_api = AgentsApi(client)
        systems_api = SystemsApi(client)

    world_state =  await init_world_state(fleet_api, agents_api, systems_api)
    wps = world_state.traits.by_wp
    nodes = build_nodes_from_traits_dict(wps)
    route_plan = plan_route_and_refuel(nodes, cur_wp, dest_wp, trav_dist, 1, 1000, max_trav_dist, 100, 0, 10, 0)
    plan_route_and_refuel()
    refuel_route = route_plan["path"]
    
    return(refuel_route)

#world_state.traits.by_wp


In [8]:
# test whether a waypoint is out of reach for a ships' fuel tank

from runtime_support import (
    setup_client_from_env,
    build_fleet_object,
    )

import asyncio

from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

with setup_client_from_env() as client:
        fleet_api = FleetApi(client)
        agents_api = AgentsApi(client)
        systems_api = SystemsApi(client)

from core_helpers import (load_initial_fleet_state,
                          init_world_state,
                          load_initial_waypoint_state,
                          get_wps_by_trait,
                          all_market_visitor,
                          api_get_ship_nav,
                          plot_route_svg,
                          patrol_markets,
                          api_navigate_ship
                          )


async def travel_checker(ship_symbol: str, waypoint_symbol: str):

    with setup_client_from_env() as client:
        fleet_api = FleetApi(client)
        systems_api = SystemsApi(client)
        agents_api = AgentsApi(client)

    world_state =  await init_world_state(fleet_api, agents_api, systems_api)

    ships = await build_fleet_object(fleet_api)
    trav_dist = ships.by_symbol[ship_symbol].fuel_current
    max_trav_dist = ships.by_symbol[ship_symbol].fuel_capacity
    cur_wp = ships.by_symbol[ship_symbol].current_waypoint

    cur_x = world_state.waypoints.by_symbol[cur_wp].x
    cur_y = world_state.waypoints.by_symbol[cur_wp].y

    dest = waypoint_symbol

    dest_x = world_state.waypoints.by_symbol[dest].x
    dest_y = world_state.waypoints.by_symbol[dest].y

    x_len = cur_x - dest_x

    y_len = cur_y - dest_y

    import math

    hypo = math.hypot(x_len, y_len)
    print("Distance between coords: ", hypo)

    if hypo > trav_dist:
        # apply routefinder to calculate best route with refuels between distant points
        print("Destination too far, initiating hops")
        route = await long_dist_travel(cur_wp, dest, trav_dist, max_trav_dist)

        for rt in route:
            await api_navigate_ship(fleet_api, ship_symbol, rt)

        return
    else:
         await api_navigate_ship(fleet_api, ship_symbol, waypoint_symbol)


#world_state.waypoints.by_symbol


In [ ]:
#await api_get_ship_nav(fleet_api, command_ship)

ShipNav(system_symbol='X1-CX40', waypoint_symbol='X1-CX40-B10', route=ShipNavRoute(destination=ShipNavRouteWaypoint(symbol='X1-CX40-B10', type=<WaypointType.ASTEROID: 'ASTEROID'>, system_symbol='X1-CX40', x=25, y=374), origin=ShipNavRouteWaypoint(symbol='X1-CX40-A4', type=<WaypointType.ORBITAL_STATION: 'ORBITAL_STATION'>, system_symbol='X1-CX40', x=8, y=26), departure_time=datetime.datetime(2025, 9, 18, 14, 58, 10, 513000, tzinfo=TzInfo(UTC)), arrival=datetime.datetime(2025, 9, 18, 15, 2, 27, 513000, tzinfo=TzInfo(UTC))), status=<ShipNavStatus.IN_ORBIT: 'IN_ORBIT'>, flight_mode=<ShipNavFlightMode.CRUISE: 'CRUISE'>)

In [ ]:
#long_dest = 'X1-CX40-J74'
#await travel_checker(command_ship, long_dest)

[BOOT] Adapted 2 ships into fleet_object
Distance between coords:  387.5061289837878
Destination too far, initiating hops


NetworkXNoPath: No path to X1-CX40-J74.

['X1-XG6-A2', 'X1-XG6-H57', 'X1-XG6-C46']
[PATROL] DDDD-2 looping through 3 markets.
[PATROL] -> Navigating to X1-XG6-A2 (#1/3)
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Prep complete
DDDD-2  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 185.937644
